# Statistical Machine Learning Rigor and Validation for Molecular Dynamics

[![Open in JupyterLab](https://img.shields.io/badge/Launch-JupyterLab-orange.svg)](https://jupyter.org)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/)

Applying conventional machine learning validation schemes (e.g., standard randomly shuffled $k$-fold cross-validation, unweighted empirical risk minimization, frame-level bootstrapping) to physical molecular dynamics data introduces **severe methodological artifacts**:

1. **Pseudo-Replication & Autocorrelation Leakage**: MD frames sampled at picosecond intervals are strongly autocorrelated. Shuffling frames across train and test folds leaks identical microstates across the partition, yielding artificially inflated accuracy.
2. **Trajectory Length Imbalance**: Unweighted loss functions allow long trajectories to dominate model parameters, biasing feature discovery toward single-trajectory noise.
3. **Overconfidence via Frame Bootstrapping**: Standard frame-level bootstrapping ignores physical correlation, generating catastrophically narrow confidence intervals.
4. **Lack of Negative Controls**: High test accuracy in complex classifiers often reflects batch confounding. Without permutation controls, models present noise as physical insights.

This notebook demonstrates the **MolScope Machine Learning Validation Framework**.


## 1. Theoretical Formulation: Autocorrelation and Effective Sample Size

In physical dynamics, consecutive frames $\mathbf{x}_t$ and $\mathbf{x}_{t+\Delta t}$ exhibit physical inertia. For any observable feature $f$, the empirical autocorrelation at lag $\tau$ is:
$$\hat{\rho}(\tau) = \frac{\sum_{t=1}^{N - \tau} (f(\mathbf{x}_t) - \bar{f})(f(\mathbf{x}_{t+\tau}) - \bar{f})}{\sum_{t=1}^N (f(\mathbf{x}_t) - \bar{f})^2}$$

The **integrated autocorrelation time** $\tau_{\text{int}}$ characterizes the physical relaxation timescale:
$$\tau_{\text{int}} = 1 + 2 \sum_{\tau=1}^{\tau_{\max}} \hat{\rho}(\tau)$$

Consequently, the **effective sample size** (ESS) of a trajectory is not $N$, but:
$$N_{\text{eff}} = \frac{N}{\tau_{\text{int}}} \ll N$$

When standard $k$-fold CV randomly splits frames, adjacent frames $t$ and $t+1$ are separated into different folds with probability $\frac{2(k-1)}{k^2}$. Because $\|\mathbf{x}_t - \mathbf{x}_{t+1}\| \approx 0$, the model evaluates on data physically identical to its training set.


## 2. Measuring Simulation Convergence & Autocorrelation


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from molscope.analysis.quality.statistical_rigor import (
    check_simulation_convergence,
    _integrated_autocorrelation_time,
)

# Simulate a synthetic feature: slow AR(1) process with high autocorrelation
np.random.seed(42)
n_frames = 2000
ar_feature = np.zeros(n_frames)
for t in range(1, n_frames):
    ar_feature[t] = 0.98 * ar_feature[t - 1] + np.random.normal(0, 0.2)

# Compute integrated autocorrelation time
tau_int = _integrated_autocorrelation_time(ar_feature, max_lag=100)
ess = n_frames / tau_int

print(f"Total Frames:                     {n_frames}")
print(f"Integrated Autocorrelation Time:  {tau_int:.1f} frames")
print(f"Effective Sample Size (ESS):      {ess:.1f} independent samples")
print(f"ESS Ratio (N_eff / N):            {ess / n_frames:.4f}")


## 3. Purged Temporal Block Cross-Validation with Embargo Margins

When target labels vary within a single trajectory (e.g., conformational states discovered via density peaks), cross-validation across trajectories is not applicable.
MolScope applies **Purged Temporal Block Cross-Validation with an Embargo Margin**:

```text
Trajectory Timeline:
[ ... Train Frames ... ][  EMBARGO  ][   TEST CHUNK   ][  EMBARGO  ][ ... Train Frames ... ]
                        <-- delta -->[ tf_start, tf_end ]<-- delta -->
```

For every test frame index $t_{\text{test}}$, any frame $t$ satisfying $|t - t_{\text{test}}| \le \Delta t_{\text{embargo}}$ is strictly purged from the training set.


In [ ]:
from molscope.analysis.predictive.workflow import purged_temporal_block_splits

# Create a sample frame index and binary target sequence
frame_labels = np.array([0] * 1000 + [1] * 1000)

splits = list(purged_temporal_block_splits(
    y=frame_labels,
    n_splits=4,
    embargo_pct=0.02,  # 2% temporal embargo buffer
))

print(f"Generated {len(splits)} Purged Block Folds:")
for fold_idx, (train_idx, test_idx) in enumerate(splits, 1):
    # Check minimum separation between test indices and train indices
    train_set = set(train_idx)
    test_set = set(test_idx)
    assert len(train_set.intersection(test_set)) == 0, "Data leakage detected!"
    print(f"  Fold {fold_idx}: Train={len(train_idx)} frames, Test={len(test_idx)} frames | Zero overlap confirmed.")


## 4. Negative Control: Permutation Null Hypothesis Testing

To prove that a trained classifier is detecting a genuine biophysical mechanism rather than trajectory batch confounding or subtle data leakage, MolScope runs an automated **permutation null test**.

Under $H_0$: $\mathbb{P}(Y \,|\, \mathbf{X}) = \mathbb{P}(Y)$.
Target labels are shuffled, destroying the physical relationship while preserving exact feature distributions, correlation structures, and frame counts.


In [ ]:
from molscope.analysis.quality.statistical_rigor import evaluate_permutation_null_control
from molscope.analysis.predictive.workflow import MLWorkflowConfig

# Generate synthetic multi-trajectory dataset: 8 trajectories
features_df = pd.DataFrame(np.random.normal(size=(800, 10)), columns=[f"F_{i}" for i in range(10)])
labels_series = pd.Series([f"state_{i % 2}" for i in range(800)])
manifest_df = pd.DataFrame({
    "trajectory_id": [f"traj_{i // 100}" for i in range(800)],
    "frame_index": range(800),
    "time_ps": range(800),
})

config = MLWorkflowConfig(supervised_splits=2, models=("logistic_regression",))

null_eval = evaluate_permutation_null_control(
    features=features_df,
    target=labels_series,
    manifest=manifest_df,
    config=config,
    n_permutations=3,
)

print("Permutation Negative Control Results:")
print(f"  Null Test Status:              {null_eval['status']}")
print(f"  Permutation Accuracies:        {null_eval['permutation_scores']}")
print(f"  Mean Null Balanced Accuracy:   {null_eval['mean_null_score']:.3f} (Chance = 0.50)")
